# Advertising Analysis and Sales Forecasting

## Multiple Linear Regression

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Read the advertising data
advertising = pd.read_csv("advertising.csv")
advertising.head()

In [ ]:
# Review the dataset
print("Dataset dimensions:", advertising.shape)
print("\nAvailable columns:")
print(advertising.columns.tolist())

print("\nMissing values:")
print(advertising.isnull().sum())

print("\nStatistical summary:")
display(advertising.describe())

## Exploring the Advertising Variables

In [ ]:
# Define the predictors and response variable
features = ["TV", "Radio", "Newspaper"]
target = "Sales"

X = advertising[features]
y = advertising[target]

print("Predictors:", features)
print("Response:", target)

In [ ]:
# Visualize each advertising channel against sales
for channel in features:
    plt.figure(figsize=(6, 4))
    plt.scatter(advertising[channel], advertising[target])
    plt.xlabel(f"{channel} Advertising Spend")
    plt.ylabel("Sales")
    plt.title(f"{channel} Advertising vs Sales")
    plt.show()

## Preparing Data for Regression

In [ ]:
# Create training and validation samples
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=10
)

print("Training observations:", X_train.shape[0])
print("Validation observations:", X_valid.shape[0])

In [ ]:
# Fit the multiple linear regression model
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print("Model intercept:", round(regressor.intercept_, 4))

for name, value in zip(features, regressor.coef_):
    print(f"{name} coefficient: {value:.4f}")

## Forecasting Sales

In [ ]:
# Generate predictions for unseen validation records
validation_predictions = regressor.predict(X_valid)

forecast = X_valid.copy()
forecast["Actual Sales"] = y_valid.to_numpy()
forecast["Estimated Sales"] = np.round(validation_predictions, 2)

display(forecast.head(10))

In [ ]:
# Estimate sales for the specified advertising plan
new_campaign = pd.DataFrame({
    "TV": [150],
    "Radio": [20],
    "Newspaper": [30]
})

sales_estimate = regressor.predict(new_campaign)[0]

print("New campaign budget:")
display(new_campaign)
print(f"Estimated sales: {sales_estimate:.2f}")

## Measuring Prediction Quality

In [ ]:
# Evaluate the regression model
mae = mean_absolute_error(y_valid, validation_predictions)
mse = mean_squared_error(y_valid, validation_predictions)
rmse = np.sqrt(mse)
r_squared = r2_score(y_valid, validation_predictions)

evaluation = pd.DataFrame({
    "Measure": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r_squared]
})

evaluation

In [ ]:
# Compare the contribution of the advertising variables
effects = pd.DataFrame({
    "Channel": features,
    "Coefficient": regressor.coef_
})

effects["Magnitude"] = effects["Coefficient"].abs()
effects.sort_values("Magnitude", ascending=False)

## Coefficient Interpretation

A regression coefficient estimates the change in Sales associated with a one-unit increase in that advertising variable while the other advertising variables remain unchanged.

The largest absolute coefficient represents the strongest estimated per-unit effect in the fitted model. The smallest absolute coefficient represents the weakest per-unit effect.

Coefficient size should be interpreted together with advertising cost and business return rather than being used as the only basis for budget allocation.

In [ ]:
# Find the channels with the largest and smallest coefficients
ordered_effects = effects.sort_values("Magnitude", ascending=False)

print("Highest estimated coefficient impact:",
      ordered_effects.iloc[0]["Channel"])
print("Lowest estimated coefficient impact:",
      ordered_effects.iloc[-1]["Channel"])

## Actual and Predicted Sales

In [ ]:
# Plot observed sales against model predictions
plt.figure(figsize=(7, 5))
plt.scatter(y_valid, validation_predictions)

lower = min(y_valid.min(), validation_predictions.min())
upper = max(y_valid.max(), validation_predictions.max())

plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual Sales vs Predicted Sales")
plt.show()

## Business Recommendation

The company should use the regression results to identify channels with a strong positive sales relationship and test whether increasing investment in those channels produces a worthwhile return. Budget decisions should also consider campaign cost and profitability, not only the coefficient size.

## Technical Improvement

The model could be improved by applying cross-validation and comparing multiple regression approaches, such as Ridge Regression, Random Forest Regression, and Gradient Boosting. This would help determine whether a nonlinear model can reduce prediction error on unseen data.

## Conclusion

Multiple Linear Regression is appropriate because Sales is a continuous numerical outcome and several advertising expenditures are used simultaneously as predictors.

The fitted model can estimate future sales for a new advertising budget and quantify the estimated relationship between each advertising channel and sales.